### RAG evaluation (have issues)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

### Download an example reference data point from LangSmith

which include questions & standard answers

In [6]:
load_dotenv()
client = Client()

In [7]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [8]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('cb1b164b-b522-475e-aad1-6e76aabff1bd'), created_at=datetime.datetime(2026, 6, 24, 5, 29, 8, 733267, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 6, 24, 5, 29, 8, 733267, tzinfo=TzInfo(0)), example_count=45, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.9.1', 'runtime_version': '3.12.3', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [11]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs

{'ground_truth': "1) Will the ACEMAGICIAN Mini PC run a specific custom Windows-only enterprise application? — requires app compatibility testing, not in chunks. 2) Will the UseBean 240W cable support a particular USB-C hub model's proprietary charging handshake? — specific hub compatibility tests not included. 3) Do the GREPHONE cables support the exact 30W fast-charge profile with my iPhone 12 while using a particular third-party charger brand? — device/charger interoperability not stated. 4) Are the NEIGHBOR 1TB flash drives pre-formatted for exFAT for use with camera recording systems? — formatting details not given. 5) Can the UPGRAVITY TV stand survive being placed on a moving boat with waves? — environmental/usage stress test data not provided.",
 'reference_context_ids': ['B0C9XFF3CT',
  'B0BP9Z159S',
  'B0BV6PWVCG',
  'B0CF57H28T',
  'B0BNN776VN'],
 'reference_descriptions': ['ACEMAGICIAN Mini PC【Dual LAN】AMD Ryzen 5 5500U, 16GB DDR4 500GB NVMe PCIe3.0 SSD Win 11 Pro【4K@60Hz T

In [12]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs

{'question': 'Which five specific user-asked questions about product compatibility or features cannot be answered with the existing chunks?'}

In [13]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs

### RAG Pipeline

In [16]:
openai_client = OpenAI()
qdrant_client = QdrantClient(url="http://localhost:6333/")


def get_embedding(text, model="text-embedding-3-small"):
    response = openai_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)
    
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    retrieved_context_ratings = []
    similarity_scores = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):
    formatted_context=""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"], 
        context["retrieved_context"], 
        context["retrieved_context_ratings"]
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt


def generate_answer(prompt):
    response = openai_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="minimal"
    )
    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }
    return final_result
    


In [17]:
rag_pipeline("Can I get some charger?", top_k=5)

{'answer': 'Yes. Here are charger options currently available:\n\n- iPhone Charger Cords (3-pack, 3 ft) – Apple MFi Certified Lightning cables\n- 5 in 1 USB C to Multi Charging Cable (10 ft) – includes Lightning, USB C, Micro USB, for multiple devices\n- MUXA 6 Pack Apple MFi Certified Lightning Cables (various lengths 3/3/6/6/10/10 ft)\n- GREPHONE USB C to Lightning Cable (6 ft) – MFi Certified, fast charging\n- Mixblu Charger Cable Replacement for Fitbit Inspire 3 (2 pack, 3.3 ft)\n\nIf you’d like, tell me how long you want the cable and for which device, and I’ll suggest a specific option.',
 'question': 'Can I get some charger?',
 'retrieved_context_ids': ['B0BYYLJRHT',
  'B0BFPZGYLD',
  'B09TNXY54Y',
  'B0BV6PWVCG',
  'B0BGDQLZD2'],
 'retrieved_context': ['iPhone Charger Cord Lightning Cables, Original 2022 Upgraded [3Pack 3ft] Apple MFi Certified USB A Charging Cable for iPhone 13 12 11 Mini Pro XR Xs Max X SE 8 7 6 Plus iPad iPod AirPods - Black Original Apple Cable - iPhone cha

### RAGAS Metrics

In [18]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

C:\Users\hyh\AppData\Local\Temp\ipykernel_23812\3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\hyh\AppData\Local\Temp\ipykernel_23812\3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\hyh\AppData\Local\Temp\ipykernel_23812\3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'raga

In [19]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\hyh\AppData\Local\Temp\ipykernel_23812\2270190839.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
C:\Users\hyh\AppData\Local\Temp\ipykernel_23812\2270190839.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [20]:
reference_input

{'question': 'Which five specific user-asked questions about product compatibility or features cannot be answered with the existing chunks?'}

In [21]:
result = rag_pipeline(reference_input["question"])

In [22]:
result

{'answer': 'Cannot answer five specific user questions about compatibility or features because the available products only provide general descriptions and do not include detailed compatibility matrices or feature specifics for each item.\n\nIf you want, I can try to infer or suggest typical questions people ask and indicate which ones are answerable from the available information versus which are not. Here are examples of five questions that would be unanswerable from the available products:\n\n- Can the 5-in-1 USB C multi charging cable reliably charge a device that requires more than 6A on the USB-C port, such as certain high-watt laptops?\n- Is the ESR iPad Air 5th/4th Gen case compatible with any third-party screen protector or stylus besides the listed models?\n- Do the MUXA 6 Pack Apple MFi certified cables support fast charging with all iPhone models beyond those explicitly listed (e.g., iPhone 15 series)?\n- Will the Garmin 890 RV GPS bundle function with vehicles that require

Faithfulness 忠实度: 生成的回答有没有瞎编（是否基于检索到的上下文）

In [23]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [24]:
await ragas_faithfulness(result, "")

0.4

Response Relevancy 回答相关性: 回答跟问题相关吗

In [25]:
async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    
    return await scorer.single_turn_ascore(sample)

In [26]:
await ragas_response_relevancy(result, "")

np.float64(0.0)

Context Precision 上下文精确率: 检索出来的 5 个商品里，有多少个是真正有用的（与标准答案对比）

In [27]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextPrecision()
    
    return await scorer.single_turn_ascore(sample)

In [28]:
await ragas_context_precision_id_based(result, reference_output)

0.2

Context Recall 上下文召回率: 标准答案里需要的 5 个商品，检索系统找回了多少个

In [29]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [30]:
await ragas_context_recall_id_based(result, reference_output)

0.2